In [ ]:
from datetime import date

import pandas as pd
import yaml
from sqlalchemy import create_engine


# Transformations

In [ ]:
dim_hora = pd.DataFrame({
    "minute_of_day": range(24 * 60)
})

dim_hora["hour"] = dim_hora["minute_of_day"] // 60
dim_hora["minute"] = dim_hora["minute_of_day"] % 60

dim_hora.head()

In [ ]:
dim_hora["hour_12"] = dim_hora["hour"].apply(lambda h: 12 if h % 12 == 0 else h % 12)
dim_hora["meridiem"] = dim_hora["hour"].apply(lambda h: "AM" if h < 12 else "PM")
dim_hora["time_str"] = dim_hora.apply(lambda r: f"{r['hour']:02d}:{r['minute']:02d}:00", axis=1)
dim_hora["time_12_str"] = dim_hora.apply(
    lambda r: f"{r['hour_12']:02d}:{r['minute']:02d} {r['meridiem']}", axis=1
)

dim_hora.head()

In [ ]:
def franja(h):
    if 5 <= h < 12:
        return "Manana"
    if 12 <= h < 18:
        return "Tarde"
    return "Noche"

dim_hora["franja_horaria"] = dim_hora["hour"].apply(franja)
dim_hora["is_business_hour"] = dim_hora["hour"].apply(lambda h: 8 <= h < 18)

dim_hora.head()

# Alinear nombres de columnas al diagrama

In [ ]:
dim_hora.rename(columns={
    'hour': 'hora',
    'minute': 'minuto',
}, inplace=True)

dim_hora = dim_hora[['minute_of_day', 'hora', 'minuto', 'hour_12', 'meridiem',
                     'time_str', 'time_12_str', 'franja_horaria', 'is_business_hour']]

dim_hora.info()
dim_hora.head()

# database connections 

In [ ]:
with open('../config.yml', 'r') as f:
    config = yaml.safe_load(f)
    config_etl = config['ETL_PRO']

url_etl = (f"{config_etl['drivername']}://{config_etl['user']}:{config_etl['password']}@{config_etl['host']}:"
           f"{config_etl['port']}/{config_etl['dbname']}")
etl_conn = create_engine(url_etl)

# load

In [ ]:
dim_hora.to_sql('dim_hora', etl_conn, if_exists='append', index_label='key_dim_hora')